# BQIS — Model Recalibration SOP
### Protokol resmi: apa yang WAJIB dilakukan begitu BQIS menerima data laboratorium baru
### (baik data pabrik asli, batch baru, atau dataset uji dari tahap kompetisi berikutnya).

**Cara pakai notebook ini (belajar bertahap):**
Jalankan SATU STEP dulu, baca outputnya, lalu tanyakan ke mentor/AI kalau ada yang
tidak dipahami SEBELUM lanjut ke step berikutnya. Jangan "Run All" langsung.

Ada 5 STEP resmi, meniru struktur audit TÜV NORD (permohonan -> seleksi -> determinasi
-> tinjauan -> keputusan) yang ada di dokumen SPC-TNI-020 kalian sendiri:

| Step BQIS | Analogi di SPC-TNI-020 |
|---|---|
| 1. Data Intake & Mapping | D. Prosedur administratif (permohonan) |
| 2. Quality Gate | D.2 Seleksi (tinjauan kelengkapan) |
| 3. Mandatory Retraining | D.3 Determinasi (evaluasi tahap 1 & 2) |
| 4. Revalidation | E. Tinjauan hasil evaluasi |
| 5. Human Sign-off & Model Card | E.2 Penetapan keputusan (oleh pihak independen) |


In [39]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
import shap
import json
from datetime import datetime

from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.cluster import KMeans
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score

np.random.seed(42)
print("Setup selesai.")

Setup selesai.


---
## STEP 1 — Data Intake & Column Mapping
**Tujuan:** memastikan data baru (apapun sumbernya) bisa dibaca sistem tanpa error,
walau nama kolomnya beda dari dataset dummy.

**Yang harus kamu lakukan sebelum run:** ganti `DATA_PATH` di bawah kalau nanti diganti
file data baru. Ganti `COLUMN_MAP` kalau nama kolom sumber baru beda.

**Setelah run, tanyakan ke aku kalau:** muncul `[WARNING] Kolom tidak ditemukan` —
itu tandanya mapping perlu disesuaikan sebelum lanjut ke Step 2.


In [40]:
DATA_PATH = "../data/bqis_biscuit_quality_dataset.csv"  # GANTI kalau ada data baru

COLUMN_MAP = {
    'sample_id': 'Sample_ID', 'batch_code': 'Batch_Code', 'product_name': 'Product_Name',
    'test_date': 'Test_Date', 'moisture': 'Moisture_Content_%', 'fat': 'Fat_Content_%',
    'protein': 'Protein_Content_%', 'water_activity': 'Water_Activity_Aw',
    'acid_insol_ash': 'Acid_Insoluble_Ash_%', 'acid_value': 'Acid_Value_mgKOHg',
    'peroxide': 'Peroxide_Value', 'tpc': 'Total_Plate_Count_CFUg',
    'yeast_mold': 'Yeast_Mold_Count_CFUg', 'lead': 'Lead_Pb_mgkg', 'cadmium': 'Cadmium_Cd_mgkg',
    'status': 'Historical_Status', 'failure_category': 'Failure_Category',
}
NUMERIC_FEATURES_STD = ['moisture', 'fat', 'protein', 'water_activity', 'acid_insol_ash',
                         'acid_value', 'peroxide', 'tpc', 'yeast_mold', 'lead', 'cadmium']

def load_and_standardize(path, column_map=COLUMN_MAP):
    df_raw = pd.read_csv(path)
    reverse_map = {v: k for k, v in column_map.items() if v is not None}
    df_std = df_raw.rename(columns=reverse_map)
    missing = [k for k in column_map if k not in df_std.columns and k != 'failure_category']
    if missing:
        print(f"[WARNING] Kolom tidak ditemukan: {missing} -- perbaiki COLUMN_MAP sebelum lanjut.")
    else:
        print(f"[OK] Semua kolom termapping dengan benar. {len(df_std)} baris dimuat.")
    return df_std

df_intake = load_and_standardize(DATA_PATH)
df_intake.head(15)

[OK] Semua kolom termapping dengan benar. 1000 baris dimuat.


,sample_id,batch_code,product_name,test_date,moisture,fat,protein,water_activity,acid_insol_ash,acid_value,peroxide,tpc,yeast_mold,lead,cadmium,failure_category,status
0,SPL-2026-0409,B-250101-4,Wafer Vanilla,2025-01-01,3.23,19.10,9.95,0.543,0.058,1.61,1.34,5670,2232,0.045,0.098,NaN,Pass
1,SPL-2026-0949,B-250101-5,Wafer Vanilla,2025-01-01,1.23,17.81,4.76,0.375,0.045,1.13,1.05,5847,272,0.014,0.018,NaN,Pass
2,SPL-2026-0482,B-250101-7,Cookies Choco Chip,2025-01-01,3.69,16.93,NaN,0.620,0.058,0.68,1.07,27579,162,0.363,0.162,NaN,Pass
3,SPL-2026-0488,B-250101-1,Cracker Plain,2025-01-01,2.69,11.10,7.07,0.505,0.071,1.69,1.27,1447,99,0.015,0.001,NaN,Pass
4,SPL-2026-0996,B-250102-6,Cracker Filled,2025-01-02,3.69,26.64,7.41,0.557,0.015,1.89,1.80,780,1397,0.509,0.039,Heavy_Metal,Fail
5,SPL-2026-0739,B-250102-4,Cookies Choco Chip,2025-01-02,3.38,NaN,5.54,0.585,0.028,1.39,1.25,47179,238,0.040,0.018,NaN,Pass
6,SPL-2026-0913,B-250102-4,Sandwich Biscuit,2025-01-02,3.49,14.02,6.39,0.519,0.029,0.48,NaN,8787,2050,0.290,0.108,NaN,Pass
7,SPL-2026-0237,B-250103-2,Cracker Filled,2025-01-03,3.13,15.55,7.65,0.547,0.061,0.72,1.24,2960,550,0.078,0.017,NaN,Pass
8,SPL-2026-0294,B-250103-6,Sandwich Biscuit,2025-01-03,3.03,20.65,4.21,0.527,0.005,0.99,1.10,6682,1129,0.075,0.022,Physicochemical,Fail
9,SPL-2026-0550,B-250103-2,Cracker Filled,2025-01-03,4.23,22.08,6.98,0.652,0.050,0.24,0.69,50742,3624,0.054,0.044,Microbiological,Fail


---
## STEP 2 — Quality Gate (Missing, Outlier, Duplicate)
**Tujuan:** sampel yang kualitas datanya jelek TIDAK boleh ikut melatih/divalidasi model
secara otomatis -- harus lolos gerbang kualitas dulu, sesuai proposal Layer 1.

**Setelah run, catat 3 angka ini dan tanyakan ke aku:**
1. Berapa % sampel di-flag manual review (missing >30%)?
2. Berapa % sampel kena flag outlier ekstrem?
3. Berapa banyak duplikat ditemukan?

Kalau salah satu angka ini TIDAK WAJAR (misal >50% ke-flag), itu sinyal data sumbernya
bermasalah serius -- JANGAN lanjut ke Step 3 sebelum investigasi kenapa.


In [41]:
def quality_gate(df, numeric_features=NUMERIC_FEATURES_STD, missing_threshold=0.30,
                  outlier_factor=3.0, id_col='sample_id'):
    df = df.copy()
    report = {}

    # 2a. Duplicate check
    if id_col in df.columns:
        n_dup = df[id_col].duplicated().sum()
        df = df.drop_duplicates(subset=id_col, keep='first').reset_index(drop=True)
        report['duplicates_removed'] = int(n_dup)

    # 2b. Missing threshold flag
    missing_pct = df[numeric_features].isna().sum(axis=1) / len(numeric_features)
    df['flag_manual_review'] = missing_pct > missing_threshold
    report['flagged_missing_pct'] = round(df['flag_manual_review'].mean() * 100, 1)

    # 2c. Outlier flag (IQR, tidak dibuang otomatis)
    df['outlier_flag'] = False
    for col in numeric_features:
        if col not in df.columns:
            continue
        q1, q3 = df[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        lower, upper = q1 - outlier_factor * iqr, q3 + outlier_factor * iqr
        df['outlier_flag'] |= ((df[col] < lower) | (df[col] > upper)).fillna(False)
    report['flagged_outlier_pct'] = round(df['outlier_flag'].mean() * 100, 1)

    df_clean = df[~df['flag_manual_review']].reset_index(drop=True)

    print("=== Quality Gate Report ===")
    print(f"Duplikat dihapus       : {report['duplicates_removed']}")
    print(f"Di-flag missing >30%   : {report['flagged_missing_pct']}%")
    print(f"Di-flag outlier ekstrem: {report['flagged_outlier_pct']}% (tetap di data, hanya ditandai)")
    print(f"\nSampel siap Step 3 (retraining): {len(df_clean)}/{len(df)}")

    return df_clean, report

df_gated, quality_report = quality_gate(df_intake)

=== Quality Gate Report ===
Duplikat dihapus       : 0
Di-flag missing >30%   : 0.0%
Di-flag outlier ekstrem: 13.1% (tetap di data, hanya ditandai)

Sampel siap Step 3 (retraining): 1000/1000


---
## STEP 3 — Mandatory Retraining (BUKAN Reuse Model Lama)
**Prinsip penting:** model XGBoost dan feature selection HARUS dilatih ulang dari nol
di atas data baru. Model yang dilatih dari data dummy TIDAK BOLEH dipakai langsung untuk
memprediksi data pabrik asli tanpa retraining -- itu prinsip yang wajib ditulis di SOP.

**Setelah run, catat:** accuracy/F1 classifier BARU ini. Bandingkan dengan hasil dummy
sebelumnya (Accuracy 0.98). Kalau turun jauh -- itu WAJAR dan JUJUR, bukan kegagalan,
justru itu tanda datanya lebih realistis dari dummy.


In [ ]:
from sklearn.model_selection import train_test_split

def retrain_classifier(df_clean, numeric_features=NUMERIC_FEATURES_STD, test_size=0.2):
    imputer = KNNImputer(n_neighbors=5, weights='distance')
    df_model = df_clean.copy()
    df_model[numeric_features] = imputer.fit_transform(df_model[numeric_features])
    df_model['status_bin'] = df_model['status'].map({'Pass': 0, 'Fail': 1})

    X = df_model[numeric_features]
    y = df_model['status_bin']

    # SPLIT DI AWAL
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=42
    )

    model = xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                               eval_metric='logloss', random_state=42)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    acc_cv = cross_val_score(model, X_train, y_train, cv=skf, scoring='accuracy')
    f1_cv = cross_val_score(model, X_train, y_train, cv=skf, scoring='f1')

    model.fit(X_train, y_train)
    explainer = shap.TreeExplainer(model)

    # BARU sekarang X_test dibuka -- evaluasi final, sekali saja
    from sklearn.metrics import accuracy_score, f1_score
    y_pred_test = model.predict(X_test)
    acc_test = accuracy_score(y_test, y_pred_test)
    f1_test = f1_score(y_test, y_pred_test)

    print(f"CV Report (train set, {len(X_train)} sampel)")
    print(f"Accuracy (5-fold): {acc_cv.mean():.4f} ± {acc_cv.std():.4f}")
    print(f"F1-Score (5-fold): {f1_cv.mean():.4f} ± {f1_cv.std():.4f}")

    print(f"\nHELD-OUT TEST Report ({len(X_test)} sampel, TIDAK PERNAH dilihat model)")
    print(f"Accuracy (test)  : {acc_test:.4f}")
    print(f"F1-Score (test)  : {f1_test:.4f}")

    if abs(acc_cv.mean() - acc_test) > 0.05:
        print("\n[WARNING] Selisih CV vs held-out test cukup besar -- indikasi overfitting/bias seleksi.")
    else:
        print("\n[OK] CV dan held-out test konsisten -- model tidak overfit ke proses seleksi.")

    report = {
        'cv_accuracy_mean': round(acc_cv.mean(),4), 'cv_accuracy_std': round(acc_cv.std(),4),
        'test_accuracy': round(acc_test,4), 'test_f1': round(f1_test,4),
        'n_train': len(X_train), 'n_test': len(X_test),
    }
    return model, explainer, X_train, y_train, report

In [43]:
model_new, explainer_new, X_train_new, y_train_new, classifier_report = retrain_classifier(df_gated)

CV Report (train set, 800 sampel)
Accuracy (5-fold): 0.9638 ± 0.0025
F1-Score (5-fold): 0.9467 ± 0.0043

HELD-OUT TEST Report (200 sampel, TIDAK PERNAH dilihat model)
Accuracy (test)  : 0.9800
F1-Score (test)  : 0.9710

[OK] CV dan held-out test konsisten -- model tidak overfit ke proses seleksi.


---
## STEP 4 — Revalidation (Clustering Ulang, Bukan Warisan Angka Lama)
**Tujuan:** ARI/NMI/purity dari dummy (0.398/0.448) TIDAK otomatis berlaku di data baru.
Feature selection dan clustering harus dihitung ulang dari nol.


In [44]:
def revalidate_clustering(df_gated, X_new, y_new, numeric_features=NUMERIC_FEATURES_STD, top_n=5):
    fail_mask = y_new == 1
    X_fail = X_new.loc[fail_mask].reset_index(drop=True)

    has_gt = 'failure_category' in df_gated.columns and df_gated['failure_category'].notna().any()
    if has_gt:
        cat_ref = df_gated.loc[fail_mask, 'failure_category'].reset_index(drop=True)
        mi_scores = mutual_info_classif(X_fail, cat_ref, random_state=42)
        ranking = pd.Series(mi_scores, index=numeric_features).sort_values(ascending=False)
    else:
        cat_ref = None
        scaler_tmp = StandardScaler()
        ranking = pd.Series(scaler_tmp.fit_transform(X_fail).var(axis=0), index=numeric_features).sort_values(ascending=False)

    selected = ranking.head(top_n).index.tolist()
    X_sel_scaled = StandardScaler().fit_transform(X_fail[selected])

    sil_scores = {}
    for k in range(2, 7):
        labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X_sel_scaled)
        sil_scores[k] = silhouette_score(X_sel_scaled, labels)
    best_k = max(sil_scores, key=sil_scores.get)
    final_labels = KMeans(n_clusters=best_k, random_state=42, n_init=10).fit_predict(X_sel_scaled)

    print(f"Fitur terpilih: {selected}")
    print(f"K-Means: k={best_k}, silhouette={sil_scores[best_k]:.3f}")

    result = {'selected_features': selected, 'best_k': best_k, 'silhouette': round(sil_scores[best_k],3)}
    if has_gt:
        ari = adjusted_rand_score(cat_ref, final_labels)
        nmi = normalized_mutual_info_score(cat_ref, final_labels)
        print(f"ARI: {ari:.3f} | NMI: {nmi:.3f}  (baseline dummy: ARI=0.398, NMI=0.448)")
        result.update({'ari': round(ari,3), 'nmi': round(nmi,3)})
    else:
        print("[INFO] Ground truth tidak tersedia -- hanya silhouette yang bisa dilaporkan.")

    return result

clustering_report = revalidate_clustering(df_gated, X_new, y_new)

Fitur terpilih: ['acid_value', 'cadmium', 'lead', 'yeast_mold', 'peroxide']
K-Means: k=5, silhouette=0.321
ARI: 0.398 | NMI: 0.448  (baseline dummy: ARI=0.398, NMI=0.448)


---
## STEP 5 — Human Sign-off & Model Card
**Prinsip:** hasil model BARU tidak otomatis dipakai produksi. Harus direview manusia
(food scientist/auditor) dulu -- sesuai janji "does not replace certification decisions"
di proposal. Cell ini men-generate ringkasan otomatis ("Model Card") sebagai bahan review.

**Setelah run, ini yang jadi dokumen resmi** yang bisa dilampirkan tiap kali BQIS
di-retrain -- baik untuk kompetisi, maupun (kalau nanti beneran dipakai) untuk audit
internal TÜV NORD.


In [45]:
model_card = {
    "generated_at": datetime.now().isoformat(),
    "data_source": DATA_PATH,
    "quality_gate": quality_report,
    "classifier": classifier_report,
    "clustering": clustering_report,
    "sign_off_status": "PENDING_HUMAN_REVIEW",
    "reviewer": None,
    "review_date": None,
    "approved_for_production": False,
}

print(json.dumps(model_card, indent=2))

with open("model_card_latest.json", "w") as f:
    json.dump(model_card, f, indent=2)
print("\nSaved: model_card_latest.json")
print("\n>> Model TIDAK dianggap 'siap pakai' sampai field 'approved_for_production'")
print(">> diubah manual jadi true oleh reviewer manusia yang berwenang.")

{
  "generated_at": "2026-07-03T17:14:21.891146",
  "data_source": "../data/bqis_biscuit_quality_dataset.csv",
  "quality_gate": {
    "duplicates_removed": 0,
    "flagged_missing_pct": 0.0,
    "flagged_outlier_pct": 13.1
  },
  "classifier": {
    "cv_accuracy_mean": 0.9638,
    "cv_accuracy_std": 0.0025,
    "test_accuracy": 0.98,
    "test_f1": 0.971,
    "n_train": 800,
    "n_test": 200
  },
  "clustering": {
    "selected_features": [
      "acid_value",
      "cadmium",
      "lead",
      "yeast_mold",
      "peroxide"
    ],
    "best_k": 5,
    "silhouette": 0.321,
    "ari": 0.398,
    "nmi": 0.448
  },
  "sign_off_status": "PENDING_HUMAN_REVIEW",
  "reviewer": null,
  "review_date": null,
  "approved_for_production": false
}

Saved: model_card_latest.json

>> Model TIDAK dianggap 'siap pakai' sampai field 'approved_for_production'
>> diubah manual jadi true oleh reviewer manusia yang berwenang.
